In [1]:
# setting the environment variables, the keys
import sys
import os

sys.path.insert(0, os.path.abspath('..'))

from config import set_environment
# for the keys - as explained early in chapter 2
set_environment()

# LangChain Common Expression Language (LCEL)

In [2]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_google_genai import GoogleGenerativeAI

# Create components
prompt = PromptTemplate.from_template("Tell me a joke aout {topic}")
# llm = ChatOpenAI()
llm = GoogleGenerativeAI(model="gemini-2.5-flash")
output_parser = StrOutputParser()

# Chain them together using LCEL
chain = prompt | llm | output_parser

# Result from the chain
result = chain.invoke({"topic": "programming"})
print(result)

An optimist sees the glass as half full.
A pessimist sees the glass as half empty.
A programmer sees the glass as twice as large as it needs to be.


# More complex expressions

In [3]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

chat = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# FIRST CHAIN - Generate story
story_prompt = PromptTemplate.from_template("Write a short story about {topic}")
story_chain = story_prompt | llm | StrOutputParser()

# SECOND CHAIN - Analyse the story
analysis_prompt = PromptTemplate.from_template(
    "Analyse the following story's mood:\n{story}"
)
analysis_chain = analysis_prompt | llm | StrOutputParser()

# Combine
story_with_analysis = story_chain | analysis_chain

# Run combined chain 
result = story_with_analysis.invoke({"topic": "a rainy day"})
print(result)

The mood of this story is predominantly one of **calm, tranquil introspection and cozy serenity, tinged with a gentle melancholy that ultimately resolves into profound peace.**

Here's a breakdown of how this mood is established:

1.  **Initial Subdued Atmosphere:** The opening sets a slightly pensive tone with "hesitant drops" and a "bruised, heavy grey" sky. This isn't overtly sad, but rather a quiet, subdued backdrop, hinting at a release or a shift.

2.  **Sensory Immersion and Softening:**
    *   **Sound:** The "gentle tapping" evolving into a "steady drumming" and "relentless rhythm" creates a pervasive, almost hypnotic soundscape. It's not harsh or alarming, but a constant, lulling presence.
    *   **Sight:** The world "blurred, softened by a liquid veil." The "vibrant green... muted into a deeper, richer hue," and the "bustling pavement was suddenly deserted." These descriptions evoke a sense of the world slowing down, becoming less demanding, and visually less sharp. The "di

## Preserving context through the chain - RunnablePassthough

In [8]:
# Using RunnablePassthrough.assign
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnablePassthrough

# Combine
enhanced_chain = RunnablePassthrough.assign(story=story_chain).assign(analysis=analysis_chain)

# Run combined chain
result = enhanced_chain.invoke({"topic": "a rainy day"})
print(result)

{'topic': 'a rainy day', 'story': 'The world outside Elara’s window was a study in soft grey. Rain, a steady, insistent drumbeat, had been falling since before dawn, transforming the familiar street into a glistening, blurred watercolour. Each droplet seemed to race its neighbour down the glass, creating shimmering, fleeting patterns.\n\nElara, wrapped in her thickest wool socks and a well-loved cardigan, nursed a mug of steaming Earl Grey. The aroma of bergamot mingled with the damp earth smell that somehow permeated the closed windows. It was the kind of day that whispered, rather than shouted; a day for introspection and quiet comfort.\n\nShe watched as the puddles in the street grew, reflecting the leaden sky like polished obsidian. A lone robin, puffed up and defiant, hopped on the lawn, its feathers matted with water, before darting under the shelter of a rhododendron bush. The leaves on the old oak tree outside glistened, each one holding a tiny bead of water, ready to surrender

## More control over the output - Construct dictionaries manually

In [9]:
from operator import itemgetter


manual_chain = (
    RunnablePassthrough() | # Pass through input
    {
        "story": story_chain, # Add stroy result
        "original_topic": itemgetter("topic"), # Preserve original topic
        "topic": itemgetter("topic") # Preserve original topic
    } | 
    RunnablePassthrough().assign(
        analysis=analysis_chain
    )
)

result = manual_chain.invoke({"topic": "a rainy day"})
print(result.keys())

dict_keys(['story', 'original_topic', 'topic', 'analysis'])


We can simplify the above with dictionary conversion using LCEL shorthand

## Simplified dictionary construction

In [12]:
simple_dict_chain_corrected = story_chain | {
    "story": RunnablePassthrough(),
    "analysis": analysis_chain
}

# analysis_chain will receive {'story':'the actual story content'} as expected.
result_corrected = simple_dict_chain_corrected.invoke({"topic": "a rainy day"})
print(result_corrected.keys())

dict_keys(['story', 'analysis'])


In [14]:
# With topic preservation
simple_dict_chain_corrected = (
    RunnablePassthrough()
    | {
        "topic": itemgetter("topic"),
        "story": story_chain,
    }
    | {
        "story": RunnablePassthrough(),
        "topic": itemgetter("topic"),
        "analysis": analysis_chain,
    }
)

# analysis_chain will receive {'story':'the actual story content'} as expected.
result_corrected = simple_dict_chain_corrected.invoke({"topic": "a rainy day"})
print(result_corrected.keys())

dict_keys(['story', 'topic', 'analysis'])


In [15]:
result["topic"]

'a rainy day'

Alternatively you can do this approach

In [ ]:
simplified_chain = {
    # Run these two in parallel
    "story": story_chain,
    "topic": itemgetter("topic"),
} | RunnablePassthrough.assign(
    # Use the output of the first step to generate the analysis
    analysis=itemgetter("story")
    | analysis_chain
)

result = simplified_chain.invoke({"topic": "a rainy day"})

print(result.keys())

dict_keys(['story', 'topic', 'analysis'])


In [19]:
result["topic"]

'a rainy day'